In [ ]:
!pip install opencv-python-headless tensorflow scikit-learn

In [ ]:
import os

classes = ["with_mask", "without_mask"]
for cls in classes:
    os.makedirs(f"dataset/{cls}", exist_ok=True)

print("Folders created:")
for cls in classes:
    print(f"  dataset/{cls}/")

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {

            // ── Overlay container ──────────────────────────
            const div = document.createElement('div');
            div.style.cssText = `
                background: #1a1a1a;
                padding: 16px;
                border-radius: 10px;
                display: flex;
                flex-direction: column;
                align-items: center;
                gap: 12px;
                max-width: 520px;
            `;

            // ── Button FIRST (above video) ─────────────────
            const btn = document.createElement('button');
            btn.textContent = '📸 Click Here to Capture Photo';
            btn.style.cssText = `
                padding: 14px 32px;
                font-size: 17px;
                font-weight: bold;
                background: #4CAF50;
                color: white;
                border: none;
                border-radius: 8px;
                cursor: pointer;
                width: 100%;
            `;
            div.appendChild(btn);

            // ── Hint text ──────────────────────────────────
            const hint = document.createElement('p');
            hint.textContent = 'Position your face in frame, then click the button above';
            hint.style.cssText = 'color:#aaa; font-size:13px; margin:0;';
            div.appendChild(hint);

            // ── Video BELOW button ─────────────────────────
            const video = document.createElement('video');
            video.style.cssText = 'width:100%; border-radius:8px; border:2px solid #444;';
            video.autoplay = true;
            div.appendChild(video);

            document.body.appendChild(div);

            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            await video.play();

            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            await new Promise((resolve) => btn.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);

            stream.getVideoTracks()[0].stop();
            div.remove();

            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data = eval_js(f'takePhoto({quality})')
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

print("take_photo() ready — button will now appear ABOVE the video.")

In [ ]:
import os
import shutil

# ── Change this variable ───────────────────────────────────────
class_name = "with_mask"   # Change to "without_mask" for second round
# ──────────────────────────────────────────────────────────────

folder = f"dataset/{class_name}"
existing = len(os.listdir(folder))

print(f"Collecting for class : '{class_name}'")
print(f"Images so far        : {existing}")
print("-" * 45)
print("STEP 1: A webcam preview will appear below.")
print("STEP 2: Allow camera access if browser asks.")
print("STEP 3: Click the GREEN button to capture.")
print("-" * 45)

try:
    filename = take_photo("temp_capture.jpg")

    dest = f"{folder}/{class_name}_{existing}.jpg"
    shutil.copy(filename, dest)
    # Brighten the saved image for better training quality
    img = cv2.imread(dest)
    img_bright = cv2.convertScaleAbs(img, alpha=1.4, beta=30)
    cv2.imwrite(dest, img_bright)

    new_count = len(os.listdir(folder))
    print(f"\nSaved  : {dest}")
    print(f"Total  : {new_count}/50 images for '{class_name}'")

    if new_count < 50:
        print(f"Action : Run this cell again. ({50 - new_count} more needed)")
    else:
        print(f"Done   : 50+ images collected! Change class_name to 'without_mask' and repeat.")

except KeyboardInterrupt:
    print("\nCapture cancelled — you interrupted before clicking the button.")
    print("Run this cell again and wait for the preview to load fully before clicking.")

except Exception as e:
    print(f"\nError: {e}")
    print("Make sure Cell 3 was run first to define take_photo().")

In [ ]:
import os
import shutil

# ── Change this variable ───────────────────────────────────────
class_name = "without_mask"   # Change to "without_mask" for second round
# ──────────────────────────────────────────────────────────────

folder = f"dataset/{class_name}"
existing = len(os.listdir(folder))

print(f"Collecting for class : '{class_name}'")
print(f"Images so far        : {existing}")
print("-" * 45)
print("STEP 1: A webcam preview will appear below.")
print("STEP 2: Allow camera access if browser asks.")
print("STEP 3: Click the GREEN button to capture.")
print("-" * 45)

try:
    filename = take_photo("temp_capture.jpg")

    dest = f"{folder}/{class_name}_{existing}.jpg"
    shutil.copy(filename, dest)
    # Brighten the saved image for better training quality
    img = cv2.imread(dest)
    img_bright = cv2.convertScaleAbs(img, alpha=1.4, beta=30)
    cv2.imwrite(dest, img_bright)

    new_count = len(os.listdir(folder))
    print(f"\nSaved  : {dest}")
    print(f"Total  : {new_count}/50 images for '{class_name}'")

    if new_count < 50:
        print(f"Action : Run this cell again. ({50 - new_count} more needed)")
    else:
        print(f"Done   : 50+ images collected! Change class_name to 'without_mask' and repeat.")

except KeyboardInterrupt:
    print("\nCapture cancelled — you interrupted before clicking the button.")
    print("Run this cell again and wait for the preview to load fully before clicking.")

except Exception as e:
    print(f"\nError: {e}")
    print("Make sure Cell 3 was run first to define take_photo().")

In [ ]:
import os

print("Dataset summary:")
print("-" * 30)
total = 0
for cls in ["with_mask", "without_mask"]:
    count = len(os.listdir(f"dataset/{cls}"))
    status = "✓ Ready" if count >= 50 else f"✗ Need {50 - count} more"
    print(f"  {cls}: {count} images  {status}")
    total += count
print("-" * 30)
print(f"  Total: {total} images")

In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# ── Load & preprocess ──────────────────────────────────────────
IMG_SIZE = 64
CATEGORIES = ["with_mask", "without_mask"]

data, labels = [], []

for label, category in enumerate(CATEGORIES):
    folder_path = os.path.join("dataset", category)
    files_list = os.listdir(folder_path)
    print(f"Loading {len(files_list)} images from '{category}'...")
    for filename in files_list:
        img_path = os.path.join(folder_path, filename)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            data.append(img)
            labels.append(label)
        else:
            print(f"  Warning: Could not read {filename}, skipping.")

X = np.array(data) / 255.0
y = to_categorical(labels)

print(f"\nTotal images loaded : {len(X)}")

# ── Train / Test split ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=labels)

print(f"Training samples    : {len(X_train)}")
print(f"Testing  samples    : {len(X_test)}")

# ── Build CNN ──────────────────────────────────────────────────
model = Sequential([
    # Block 1 — detect edges and basic features
    Conv2D(32, (3, 3), activation='relu',
           input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2, 2),

    # Block 2 — detect complex patterns
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    # Classifier head
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(2, activation='softmax')   # [with_mask, without_mask]
])

model.summary()

# ── Compile & Train ────────────────────────────────────────────
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

print("\nTraining started...")
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=16,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

# ── Save model ─────────────────────────────────────────────────
model.save("mask_detector.keras")
print("\nModel saved as 'mask_detector.keras'")

# ── Plot results ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val accuracy')
axes[0].set_title('Accuracy over epochs')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train loss')
axes[1].plot(history.history['val_loss'], label='Val loss')
axes[1].set_title('Loss over epochs')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

# ── Final evaluation ───────────────────────────────────────────
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest accuracy : {accuracy * 100:.2f}%")
print(f"Test loss     : {loss:.4f}")

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from IPython.display import display
from IPython.display import Image as IPImage

# ── Load model ─────────────────────────────────────────────────
model = load_model("mask_detector.keras")
labels_map = {0: "Mask", 1: "No Mask"}
colors_map  = {0: (0, 255, 0), 1: (0, 0, 255)}   # Green / Red
IMG_SIZE = 64

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

if face_cascade.empty():
    raise IOError("Haar cascade file not found.")

print("Webcam will open — position your face and click 'Capture Photo'.")
print("Run this cell again to capture another frame.\n")

# ── Capture frame ──────────────────────────────────────────────
image_file = take_photo('live_frame.jpg')

# ── Run detection ──────────────────────────────────────────────
frame = cv2.imread(image_file)

if frame is None:
    print("Error: Could not read captured image.")
else:
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(60, 60)
    )

    if len(faces) == 0:
        print("No faces detected in this frame.")
        print("Tips: improve lighting, move closer, face the camera directly.")
    else:
        print(f"Detected {len(faces)} face(s).\n")

    for (x, y, w, h) in faces:
        # Crop, resize, normalize
        face_img    = frame[y:y+h, x:x+w]
        face_resized = cv2.resize(face_img, (IMG_SIZE, IMG_SIZE))
        face_norm   = face_resized / 255.0
        face_input  = np.reshape(face_norm, (1, IMG_SIZE, IMG_SIZE, 3))

        # Predict
        pred        = model.predict(face_input, verbose=0)[0]
        class_idx   = np.argmax(pred)
        confidence  = pred[class_idx] * 100
        label_text  = f"{labels_map[class_idx]}: {confidence:.1f}%"
        color       = colors_map[class_idx]

        print(f"  Face at ({x},{y}) → {label_text}")

        # Draw bounding box and label
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.rectangle(frame, (x, y-30), (x+w, y), color, -1)   # label bg
        cv2.putText(frame, label_text, (x+4, y-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

    # ── Display result ─────────────────────────────────────────
    _, buffer = cv2.imencode('.jpg', frame)
    display(IPImage(data=buffer.tobytes()))

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from IPython.display import display
from IPython.display import Image as IPImage

# ── Load model ─────────────────────────────────────────────────
model = load_model("mask_detector.keras")
labels_map = {0: "Mask", 1: "No Mask"}
colors_map  = {0: (0, 255, 0), 1: (0, 0, 255)}   # Green / Red
IMG_SIZE = 64

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

if face_cascade.empty():
    raise IOError("Haar cascade file not found.")

print("Webcam will open — position your face and click 'Capture Photo'.")
print("Run this cell again to capture another frame.\n")

# ── Capture frame ──────────────────────────────────────────────
image_file = take_photo('live_frame.jpg')

# ── Run detection ──────────────────────────────────────────────
frame = cv2.imread(image_file)

if frame is None:
    print("Error: Could not read captured image.")
else:
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(60, 60)
    )

    if len(faces) == 0:
        print("No faces detected in this frame.")
        print("Tips: improve lighting, move closer, face the camera directly.")
    else:
        print(f"Detected {len(faces)} face(s).\n")

    for (x, y, w, h) in faces:
        # Crop, resize, normalize
        face_img    = frame[y:y+h, x:x+w]
        face_resized = cv2.resize(face_img, (IMG_SIZE, IMG_SIZE))
        face_norm   = face_resized / 255.0
        face_input  = np.reshape(face_norm, (1, IMG_SIZE, IMG_SIZE, 3))

        # Predict
        pred        = model.predict(face_input, verbose=0)[0]
        class_idx   = np.argmax(pred)
        confidence  = pred[class_idx] * 100
        label_text  = f"{labels_map[class_idx]}: {confidence:.1f}%"
        color       = colors_map[class_idx]

        print(f"  Face at ({x},{y}) → {label_text}")

        # Draw bounding box and label
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.rectangle(frame, (x, y-30), (x+w, y), color, -1)   # label bg
        cv2.putText(frame, label_text, (x+4, y-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

    # ── Display result ─────────────────────────────────────────
    _, buffer = cv2.imencode('.jpg', frame)
    display(IPImage(data=buffer.tobytes()))

In [ ]:
import zipfile
import os
from google.colab import files

with zipfile.ZipFile("mask_detection_project.zip", "w") as zipf:
    # Notebook — you'll add this manually
    # Dataset
    for cls in ["with_mask", "without_mask"]:
        folder = f"dataset/{cls}"
        if os.path.exists(folder):
            for fname in os.listdir(folder):
                zipf.write(f"{folder}/{fname}")

    # Model
    if os.path.exists("mask_detector.keras"):
        zipf.write("mask_detector.keras")

    # Haar cascade isn't needed — it comes with OpenCV

print("Zip created!")
files.download("mask_detection_project.zip")